<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B08%5D%20-%20Ingenieria_de_Variables_II/%5B01%5D%20-%20Notebooks/E4_Caza_la_Fuga_de_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E4 · Caza la fuga de PCA - Ingeniería de Variables II (bonus)

## Introducción

> **Leakage (fuga de datos)**: el modelo usa información que no tendría al predecir de verdad.
> Resultado: score buenísimo en pruebas y decepción en producción.

Cualquier paso que **aprenda** de los datos (escalado, PCA, selección...) debe ajustarse
**solo con train**. La pregunta de hoy: ¿cuánta fuga mete el PCA si lo ajustas antes del
split? Y, sobre todo, ¿qué pasos filtran de verdad?

- **Parte A**: PCA antes del split vs PCA dentro del Pipeline. PCA es **extracción no
  supervisada** (no mira el target).
- **Parte B**: una selección de variables **que sí mira el target**, hecha antes del split.
  Aquí la fuga es enorme.

## Objetivos del ejercicio

- Medir la fuga del **PCA** (paso no supervisado) con validación cruzada.
- Ver una fuga **grande** de verdad: selección de variables por el target antes del split.
- Quedarte con la regla de oro: **todo en el Pipeline, fit solo en train**.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

### Parte A · ¿Filtra el PCA? (extracción no supervisada)

Comparamos dos formas con **validación cruzada** (más fiable que un único split):

- **MAL**: escalar y aplicar PCA sobre **todos** los datos y luego validar.
- **BIEN**: meter escalado y PCA **dentro del Pipeline**, que en cada *fold* se ajusta solo
  con su train.

In [ ]:
import numpy as np
import pandas as pd

def generar_datos_anchos(n=800, n_cols=60, n_latentes=5, n_clases=2,
                         separacion=2.5, ruido=0.6, semilla=42):
    # Genera un dataset "ancho": muchas columnas (sensores) pero POCAS dimensiones
    # reales. Unos pocos factores latentes generan casi toda la informacion; el
    # resto de columnas son mezclas de esos factores mas ruido. Asi PCA puede
    # recuperar la estructura con pocas componentes.
    rng = np.random.default_rng(semilla)

    # Centros de cada clase en el espacio latente (grupos separados)
    centros = rng.normal(scale=separacion, size=(n_clases, n_latentes))
    y = rng.integers(0, n_clases, size=n)
    Z = centros[y] + rng.normal(size=(n, n_latentes))      # factor latente de cada fila

    # Cada columna observada = mezcla ponderada de los factores latentes + ruido
    cargas = rng.normal(size=(n_latentes, n_cols))
    X = Z @ cargas + ruido * rng.normal(size=(n, n_cols))

    # Escalas MUY distintas por columna (para motivar StandardScaler antes de PCA)
    escalas = rng.uniform(1, 1000, size=n_cols)
    X = X * escalas

    cols = [f"sensor_{i:03d}" for i in range(n_cols)]
    df = pd.DataFrame(X, columns=cols)
    df["target"] = y
    return df

In [ ]:
df = generar_datos_anchos(n=150, n_cols=800, n_latentes=10,
                          separacion=0.7, ruido=1.5, semilla=3)
X = df.drop(columns="target")
y = df["target"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# MAL: escalado + PCA ajustados con TODO el dataset, y despues validamos
X_esc_todo = StandardScaler().fit_transform(X)
X_pca_todo = PCA(n_components=40, random_state=0).fit_transform(X_esc_todo)
auc_mal = cross_val_score(LogisticRegression(max_iter=1500),
                          X_pca_todo, y, cv=cv, scoring="roc_auc").mean()

# BIEN: todo dentro del Pipeline (se reajusta en cada fold solo con su train)
pipe = Pipeline([
    ("escalar", StandardScaler()),
    ("pca", PCA(n_components=40, random_state=0)),
    ("clf", LogisticRegression(max_iter=1500)),
])
auc_bien = cross_val_score(pipe, X, y, cv=cv, scoring="roc_auc").mean()

print(f"[MAL]  PCA antes del split   -> AUC CV: {auc_mal:.3f}")
print(f"[BIEN] PCA dentro del Pipeline-> AUC CV: {auc_bien:.3f}")
print(f"Diferencia: {auc_mal - auc_bien:+.3f}  (muy pequeña)")

La diferencia es **mínima**. ¿Por qué? Porque **PCA no mira el target**: ajustarlo con
train+test cambia un poco las direcciones, pero no le "chiva" las etiquetas. La fuga de un
paso no supervisado suele ser pequeña. Aun así, lo correcto es ajustarlo solo con train.

### Parte B · La fuga de verdad: seleccionar por el target antes del split

Ahora un paso **supervisado**: elegir las variables más correlacionadas con el target. Para
que se vea claro, usamos datos de **puro ruido** y un target **aleatorio**: no existe ninguna
señal real, así que un modelo honesto debe dar **AUC ~0.5**.

In [ ]:
rng = np.random.default_rng(0)
n, p, k = 200, 2000, 20
X_ruido = rng.normal(size=(n, p))     # puro ruido, sin relación con el target
y_az = rng.integers(0, 2, size=n)     # target aleatorio
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# MAL: elegir las k columnas mas correlacionadas con y usando TODOS los datos, y luego validar
corr = np.array([abs(np.corrcoef(X_ruido[:, j], y_az)[0, 1]) for j in range(p)])
top = np.argsort(corr)[::-1][:k]
auc_mal = cross_val_score(LogisticRegression(max_iter=1000),
                          X_ruido[:, top], y_az, cv=cv, scoring="roc_auc").mean()

# BIEN: la seleccion va dentro del Pipeline (en cada fold, solo con su train)
pipe = Pipeline([
    ("seleccion", SelectKBest(f_classif, k=k)),
    ("clf", LogisticRegression(max_iter=1000)),
])
auc_bien = cross_val_score(pipe, X_ruido, y_az, cv=cv, scoring="roc_auc").mean()

print(f"[MAL]  selección sobre TODO   -> AUC CV: {auc_mal:.3f}  (¡falso! no hay señal real)")
print(f"[BIEN] selección en Pipeline  -> AUC CV: {auc_bien:.3f}  (~0.5, lo honesto)")
print(f"Diferencia (fuga): {auc_mal - auc_bien:+.3f}")

### ¿Cuál tiene fuga y por qué?

- El **PCA antes del split** (Parte A) casi no filtra: es **no supervisado**, no usa el target.
- La **selección por el target antes del split** (Parte B) filtra muchísimo: hemos sacado
  "señal" de **puro ruido** porque elegimos las columnas mirando el target con todos los datos,
  incluido el test. El AUC sube a falsamente alto.

**Regla de oro**: cualquier paso que aprenda algo de los datos (sobre todo si mira el target)
va **dentro del Pipeline** y se ajusta **solo con train**. Así estás protegido tanto con
pasos no supervisados (PCA, escalado) como supervisados (selección por target).

### Reflexión

1. ¿Por qué la fuga del PCA es pequeña y la de la selección por target es enorme?
2. En la Parte B no había señal real. ¿Cómo es posible un AUC tan alto en la versión con fuga?
3. ¿Qué tienen en común todas las fugas y cómo las evita un Pipeline?
4. ¿Se te ocurre otro paso de preprocesado que filtre si lo haces antes del split?